In [10]:
import pandas as pd
import numpy as np
import re
import string
import pickle

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [11]:
df = pd.read_csv("../dataset/youtube_clean.csv")

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()

,Comment,Sentiment,comment_length
0,lets not forget that apple pay in 2014 require...,neutral,317
1,here in nz 50 of retailers don’t even have con...,negative,163
2,i will forever acknowledge this channel with t...,positive,183
3,whenever i go to a place that doesn’t take app...,negative,450
4,apple pay is so convenient secure and easy to ...,positive,135


In [12]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [13]:
from nltk.corpus import wordnet

def get_wordnet_pos(tag):

    if tag.startswith('J'):
        return wordnet.ADJ

    elif tag.startswith('V'):
        return wordnet.VERB

    elif tag.startswith('N'):
        return wordnet.NOUN

    elif tag.startswith('R'):
        return wordnet.ADV

    return wordnet.NOUN

In [14]:
def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)

    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    text = re.sub(r'\d+', '', text)

    text = text.translate(
        str.maketrans(
            '',
            '',
            string.punctuation
        )
    )

    tokens = word_tokenize(text)

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    pos_tags = pos_tag(tokens)

    tokens = [
        lemmatizer.lemmatize(
            word,
            get_wordnet_pos(tag)
        )
        for word, tag in pos_tags
    ]

    return " ".join(tokens)

In [15]:
df["clean_comment"] = df["Comment"].apply(preprocess_text)

In [16]:
df = df[
    df["clean_comment"].str.strip() != ""
]

In [17]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["Sentiment"]
)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_comment"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [19]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

In [20]:
train_df.to_csv(
    "../dataset/train.csv",
    index=False
)

test_df.to_csv(
    "../dataset/test.csv",
    index=False
)

In [ ]:
print(df.shape)
print(df["Sentiment"].value_counts())
print(train_df.shape)
print(test_df.shape)

(17868, 5)
Sentiment
positive    11052
neutral      4499
negative     2317
Name: count, dtype: int64
(14294, 2)
(3574, 2)
